# kinyarwanda tweet sentiment classification

finetuning afroxlmr on the afrisenti kinyarwanda subset against tfidf baselines

**run order matters** sections 14 are cpuonly and take about two minutes section 5
needs a gpu *runtime > change runtime type > t4 gpu* section 6 touches the test set and
must be run only once after the experiments in section 5 are finished

every configuration decision in this notebook is made on the **validation** split
the test split is evaluated exactly once in section 6

## 0 setup

In [ ]:
# on colab clone the repo and work from its root so that `from src` imports
# resolve and every output path in the modules lands inside the repo
import os, sys, subprocess

REPO_URL = "https://github.com/mbonikevin/sentiment-kinya.git"
REPO_DIR = "sentiment-kinya"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir(REPO_DIR)
    # colab ships torch; installing it again is slow and can break the cuda build
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers>=4.40,<5", "accelerate>=0.30", "datasets>=2.18",
                    "sentencepiece", "protobuf"], check=True)

sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

from src.data import load_splits, describe, save_processed, ID_TO_LABEL, LABEL_ORDER
from src.metrics import compute_metrics, format_metrics_report, save_metrics, plot_confusion_matrix, comparison_table
from src import viz, leakage
print("label mapping:", ID_TO_LABEL)

## 1 data

afrisenti muhammad et al 2023 kinyarwanda subset read from the official
afrisentisemeval2023 repository rather than the hugging face mirror the mirror is
loaded by a dataset script and `datasets` 4.0 removed script support so
`load_dataset` on it raises an error

we stay on the organisers own train/dev/test split instead of making our own so results
sit on the same footing as the published semeval2023 task 12 numbers

In [ ]:
splits = load_splits()        # light cleaning urls > <url> whitespace collapsed
save_processed(splits)
describe(splits)

### 1.1 does the official split leak?

these splits come from the organisers which is no reason to take it on trust
two lines of checking changes how you should read every score under it

In [ ]:
print(leakage.overlap_report(splits).to_string(index=False))
print()
print("accuracy ceiling implied by contradictory duplicates:",
      leakage.annotation_ceiling(splits, "test"))

contra = leakage.contradictory_examples(splits, "train", "test")
print(f"\n{len(contra)} tweets appear verbatim in train and test with DIFFERENT gold labels:")
contra.head(10)

### 1.2 figures for the report

In [ ]:
viz.class_distribution(splits)
viz.length_distribution(splits, max_len=128)
plt.show()

## 2 how the tokenizers segment kinyarwanda

people usually argue an africa adapted model segments the language better
that is testable so we test it before leaning on it

In [ ]:
from src.tokenizer_analysis import fertility_table, segmentation_examples

print(fertility_table(splits["train"]["text"].tolist(), max_len=128).to_string(index=False))
print()
print(segmentation_examples().to_string(index=False))

**read this carefully before section 5** afroxlmr and xlmr report the same vocabulary
size and the same subwordsperword because adaptive pretraining does not change the
vocabulary so any gain afroxlmr shows in experiment e5 cannot be attributed to better
tokenization  it comes from the encoder weights that is the distinction to make in
the report and the video

## 3 baselines

three of them getting stronger the majority class one is there to make accuracy
figures interpretable; the tfidf + logistic regression model is the one the transformer
actually has to beat

In [ ]:
from src.baseline import BASELINES, run_baseline, top_features

fitted, baseline_val = {}, {}
for name in BASELINES:
    model, m, secs = run_baseline(name, splits, eval_split="validation")
    fitted[name], baseline_val[name] = model, m
    print(format_metrics_report(m, name), f"\n  fit: {secs:.2f}s\n")

In [ ]:
# what the linear model keys on useful in the report these are real kinyarwanda
# sentiment words which is why the baseline is a fair opponent rather than a straw man
for label, feats in top_features(fitted["TF-IDF + LogReg"], n=10).items():
    print(f"{label:9s}: {', '.join(w for w, _ in feats)}")

## 4 baseline test scores

run once the baselines are tuned to their library defaults rather than searched so
there is nothing here that could be influenced by seeing the test set

In [ ]:
for name in BASELINES:
    model, m, _ = run_baseline(name, splits, eval_split="test")
    save_metrics(m, name, "test")
    print(f"{name:24s} macro-F1 {m['macro_f1']:.4f}  weighted-F1 {m['weighted_f1']:.4f}  acc {m['accuracy']:.4f}")

BASELINE_VAL_F1 = baseline_val["TF-IDF + LogReg"]["macro_f1"]
print(f"\nbar to beat on validation: macro-F1 {BASELINE_VAL_F1:.4f}")

## 5 transformer experiments  *gpu required*

each experiment states a hypothesis before it runs and every run is appended to
`reports/results/experiments_*.csv` with that hypothesis the result and the decision
taken validation only  no test scores anywhere in this section

| id | change | hypothesis |
|----|--------|------------|
| e1 | afroxlmrbase standard recipe | a transformer pretrained on kinyarwanda beats tfidf |
| e2 | learning rate 1e5 / 5e5 | 2e5 is the roberta convention; 5e5 may be unstable on 3.3k examples |
| e3 | 5 epochs instead of 3 | more passes help or overfit a small training set |
| e4 | max_len 64 instead of 128 | 5% of tweets truncate for roughly half the compute |
| e5 | plain xlmr identical config | **the key ablation** isolates africanlanguage adaptive pretraining |
| e6 | heavier cleaning no emoji elongation collapsed | emoji carry sentiment so removing them should hurt |
| e7 | classweighted loss | weighting the minority class raises macrof1 |

In [ ]:
import torch, transformers
print("transformers:", transformers.__version__)
assert int(transformers.__version__.split(".")[0]) < 5, (
    "src/train_transformer.py targets the transformers 4.x Trainer API. "
    "Re-run the setup cell, then Runtime > Restart session.")
assert torch.cuda.is_available(), (
    "No GPU. Runtime > Change runtime type > T4 GPU, then re-run from the top.")
print("GPU:", torch.cuda.get_device_name(0),
      f"({torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB)")

In [ ]:
from src.train_transformer import TrainConfig, run_experiment, predict, AFRO_XLMR_BASE, XLMR_BASE
from src.experiment_log import reset_log, load_all_experiments

# rerunning this notebook should not duplicate rows in the log
for m in ("afro-xlmr-base", "xlm-roberta-base"):
    reset_log(m)

runs = {}

In [ ]:
# if this raises cuda out of memory halve batch_size to 8 everything else
# stays comparable because only the optimiser step size changes not the data

# e1  the reference configuration
runs["E1"] = run_experiment(
    "E1", TrainConfig(label="E1-afroxlmr-default"), splits,
    change="AfroXLMR-base, lr 2e-5, 3 epochs, max_len 128, batch 16",
    hypothesis="A model with Kinyarwanda in its pretraining beats TF-IDF+LogReg on macro-F1",
    decision="", reason="")

In [ ]:
# e2  learning rate two runs one below one above the convention
for lr, tag in [(1e-5, "E2a"), (5e-5, "E2b")]:
    runs[tag] = run_experiment(
        tag, TrainConfig(learning_rate=lr, label=f"{tag}-lr{lr:g}"), splits,
        change=f"learning rate {lr:g} instead of 2e-5",
        hypothesis="2e-5 is the RoBERTa-family convention; 1e-5 may underfit in 3 epochs and 5e-5 may destabilise a 3.3k-example fine-tune",
        decision="", reason="")

In [ ]:
# e3  more epochs load_best_model_at_end means a worse final epoch is not kept
# so this measures whether an extra pass finds anything not whether it overfits
runs["E3"] = run_experiment(
    "E3", TrainConfig(epochs=5, label="E3-5epochs"), splits,
    change="5 epochs instead of 3",
    hypothesis="3.3k examples is small; extra epochs will overfit rather than help",
    decision="", reason="")

In [ ]:
# e4  shorter sequences section 2 measured that 5% of tweets exceed 64 subwords
runs["E4"] = run_experiment(
    "E4", TrainConfig(max_len=64, label="E4-maxlen64"), splits,
    change="max_len 64 instead of 128",
    hypothesis="Truncating ~5% of tweets costs little and roughly halves training time",
    decision="", reason="")

In [ ]:
# e5  the ablation same everything base model swapped section 2 established the
# two share a vocabulary so this difference is attributable to pretraining alone
runs["E5"] = run_experiment(
    "E5", TrainConfig(model_name=XLMR_BASE, label="E5-xlmr-ablation"), splits,
    change="plain XLM-RoBERTa-base in place of AfroXLMR-base, all else identical",
    hypothesis="AfroXLMR's African-language adaptive pretraining is worth a measurable amount of macro-F1",
    decision="", reason="")

In [ ]:
# e6  preprocessing reloads the data with the heavier cleaning flags on
splits_heavy = load_splits(strip_emoji=True, fix_elongation=True)
runs["E6"] = run_experiment(
    "E6", TrainConfig(label="E6-heavy-cleaning"), splits_heavy,
    change="emoji stripped and elongated characters collapsed",
    hypothesis="Emoji are often the only sentiment marker in a code-mixed tweet, so removing them should LOWER macro-F1",
    decision="", reason="")

In [ ]:
# e7  class weighting matching the baseline's class_weight="balanced"
runs["E7"] = run_experiment(
    "E7", TrainConfig(class_weights=True, label="E7-class-weighted"), splits,
    change="inverse-frequency class weights in the cross-entropy loss",
    hypothesis="Up-weighting the smaller positive class raises macro-F1 even if accuracy drops",
    decision="", reason="")

### 5.1 the experiment table

In [ ]:
log = load_all_experiments()
display(log[["exp_id", "stage", "change", "val_macro_f1", "val_weighted_f1", "val_accuracy", "train_seconds"]])

viz.experiment_results(log, baseline_f1=BASELINE_VAL_F1)
plt.show()

best_id = log.loc[log["val_macro_f1"].idxmax(), "exp_id"]
print(f"\nbest on validation: {best_id}")
print("Apply the tie rule from src/experiment_log.py before declaring a winner: "
      "configurations within 0.01 macro-F1 are tied, and the cheaper one wins.")

## 6 final test evaluation  *run once*

pick the winning configuration by the rule above set `BEST` to its key in `runs` then
run this section nothing after this point may feed back into a configuration choice

In [ ]:
BEST = "E1"   # < set from section 51
best = runs[BEST]
cfg, trainer, tokenizer = best["cfg"], best["trainer"], best["tokenizer"]
print("final model:", cfg.summary())

test_preds, test_probs = predict(trainer, splits, "test", cfg.max_len, tokenizer)
m_test = compute_metrics(splits["test"]["label"], test_preds)
save_metrics(m_test, f"AfroXLMR ({BEST})", "test")
print(format_metrics_report(m_test, f"AfroXLMR ({BEST})"))

In [ ]:
# leakfree score the same predictions rescored with the 27 trainduplicate rows removed
mask = leakage.clean_mask(splits, "test").to_numpy()
m_clean = compute_metrics(splits["test"]["label"][mask], test_preds[mask])
save_metrics(m_clean, f"AfroXLMR ({BEST}) clean", "test")
print(f"full test  ({len(test_preds)} rows): macro-F1 {m_test['macro_f1']:.4f}")
print(f"clean test ({int(mask.sum())} rows): macro-F1 {m_clean['macro_f1']:.4f}")
print("A large gap would mean the score leans on memorised duplicates.")

In [ ]:
plot_confusion_matrix(m_test["confusion_matrix"], f"AfroXLMR ({BEST})", normalise=True)
plt.show()
comparison_table()

## 7 error analysis

reading a few failures usually just confirms what you already thought so every
hypothesis about *why* the model fails is stated as a surface feature and tested by
comparing error rates and only the ones the data supports go in the report

In [ ]:
from src import error_analysis as ea

frame = ea.build_frame(splits["test"], test_preds, test_probs, model_name=f"AfroXLMR ({BEST})")
paths = ea.export(frame, f"afroxlmr_{BEST}")

rates = ea.feature_error_rates(frame)
display(rates)
viz.error_feature_chart(rates, 1 - frame["correct"].mean())
plt.show()

In [ ]:
print("Which classes get confused with which:")
display(ea.confusion_pairs(frame))

print("\nMost CONFIDENT errors -- what the model has learned wrongly:")
display(frame[~frame["correct"]].nlargest(10, "confidence")[
    ["raw_text", "true_label", "pred_label", "confidence"]])

print("\nMost confident CORRECT predictions -- the 'performs well' examples:")
display(frame[frame["correct"]].nlargest(5, "confidence")[
    ["raw_text", "true_label", "confidence"]])

In [ ]:
# where the transformer and the baseline disagree the most direct evidence for what
# pretraining actually bought in both directions
import numpy as np
base_preds = fitted["TF-IDF + LogReg"].predict(splits["test"]["text"])
base_probs = fitted["TF-IDF + LogReg"].predict_proba(splits["test"]["text"])
base_frame = ea.build_frame(splits["test"], base_preds, base_probs, model_name="TF-IDF + LogReg")

disagree = ea.disagreement(base_frame, frame, "baseline", "transformer")
disagree.to_csv("reports/results/disagreement_baseline_vs_transformer.csv", index=False)
display(disagree.head(12))

## 8 publish the model

pushes the finetuned weights to the hugging face hub so that the gradio app in `app/`
loads them by name the app holds no weights itself which keeps the repo small and
means the deployed space and this notebook are provably running the same model

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
HUB_ID = "mbonikevin/afroxlmr-kinyarwanda-sentiment"   # < set this

trainer.model.push_to_hub(HUB_ID)
tokenizer.push_to_hub(HUB_ID)
print(f"https://huggingface.co/{HUB_ID}")
print("Set MODEL_ID in app/app.py to this, then deploy the Space.")